# 04 - Train next-interval consumption forecast

Requires 97+ consecutive ACTUAL intervals per meter. Two days at 15-minute cadence produces 192 intervals per meter.

In [ ]:
import os, re
from datetime import datetime, timezone
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import Window, functions as F
catalog = os.getenv('AIDP_CATALOG', 'aidp_poc')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', catalog): raise ValueError('AIDP_CATALOG must be a simple Spark identifier')
silver, features_table = f'{catalog}.silver.interval_reading', f'{catalog}.ml.meter_features'
base = spark.table(silver).where("quality_code='ACTUAL' AND meter_status='ACTIVE'")
w = Window.partitionBy('meter_id').orderBy('interval_start_utc')
features = (base.select('meter_id','interval_start_utc','reading_date','consumption_kwh','temperature_c','humidity_pct','voltage_v','demand_kw','tariff_code','customer_segment').withColumn('target_next_kwh',F.lead('consumption_kwh').over(w)).withColumn('lag_1_kwh',F.lag('consumption_kwh',1).over(w)).withColumn('lag_4_kwh',F.lag('consumption_kwh',4).over(w)).withColumn('lag_96_kwh',F.lag('consumption_kwh',96).over(w)).withColumn('rolling_4_kwh',F.avg('consumption_kwh').over(w.rowsBetween(-4,-1))).withColumn('hour',F.hour('interval_start_utc')).withColumn('day_of_week',F.dayofweek('interval_start_utc')).withColumn('is_weekend',F.when(F.dayofweek('interval_start_utc').isin(1,7),1).otherwise(0)).drop('consumption_kwh').dropna().withColumn('ts_epoch',F.col('interval_start_utc').cast('long')))
if not features.take(1): raise ValueError('Insufficient history: wait until each meter has 97 valid ACTUAL 15-minute readings. The scheduled 2-day producer run provides 192.')
features.drop('ts_epoch').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(features_table)
cutoff = features.approxQuantile('ts_epoch',[.80],.01)[0]; train, test = features.where(F.col('ts_epoch') < cutoff), features.where(F.col('ts_epoch') >= cutoff)
if not train.take(1) or not test.take(1): raise ValueError('Training requires readings across more than one time split.')
numeric = ['lag_1_kwh','lag_4_kwh','lag_96_kwh','rolling_4_kwh','hour','day_of_week','is_weekend','temperature_c','humidity_pct','voltage_v','demand_kw']
pipeline = Pipeline(stages=[StringIndexer(inputCol='tariff_code',outputCol='tariff_idx',handleInvalid='keep'),StringIndexer(inputCol='customer_segment',outputCol='segment_idx',handleInvalid='keep'),VectorAssembler(inputCols=numeric+['tariff_idx','segment_idx'],outputCol='features'),GBTRegressor(labelCol='target_next_kwh',maxIter=75,maxDepth=6,seed=42)])
model = pipeline.fit(train); predictions = model.transform(test)
metrics = {m:RegressionEvaluator(labelCol='target_next_kwh',predictionCol='prediction',metricName=m).evaluate(predictions) for m in ('mae','rmse')}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); path=f'/Workspace/models/smart_meter_next_kwh/{run_id}'; model.write().overwrite().save(path)
print(f'Saved candidate model to {path}; validation metrics={metrics}. Set MODEL_URI to this path before scoring.')